# 09 — Error analysis

Analyze classification, localization, duplicates/background, misses, tiny-object misses, confidence calibration, occlusion/truncation, and density slices. Qualitative images must be selected by measured differences, not cherry-picked.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/visdrone_architecture_benchmark"
GITHUB_USERNAME = "Harryphan72007"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/aerial-object-detection-benchmark.git"
REPO_DIR = "/content/aerial-object-detection-benchmark"
!test -d {REPO_DIR}/.git || git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements-colab.txt
!pip install -q -e .

In [ ]:
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
collect_environment()

In [ ]:
import json, pandas as pd
files=sorted(paths.evaluation.glob("*__metrics.json"))
summary=[]
for f in files:
    d=json.loads(f.read_text()); summary.append({"model":d.get("model_id"),"mAP":d.get("mAP"),"APtiny":d.get("APtiny"),**d.get("error_counts",{})})
pd.DataFrame(summary)

## Automatic image selection

Use per-image TP/FP/FN and tiny-miss counts to select best, worst, largest pairwise differences, most tiny misses, and most false positives. Save a selection manifest before rendering side-by-side predictions.